# 1. Fine-tuning, una condizione per sigla

Allena un adapter per ogni condizione di cui esiste il file, e salta quelle gia' fatte.

Runtime: Colab T4

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import torch, os, glob, json, re, random, gc, shutil

assert torch.cuda.is_available(), "no GPU available"
print(
    torch.cuda.get_device_name(0),
    f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB",
)

Tesla T4 | 16 GB


## Registro e iperparametri

`USE_BF16` e' **impostato a False**.

In [ ]:
# Map<str, tuple> -> permette di associare codice a file, entry per finetuning e desc
REGISTRY = {
    "M0": ("corpus.jsonl", "text", "raw"),
    "M1": ("corpus.presidio.jsonl", "text_anon", "presidio"),
    "M2": ("corpus.llm.jsonl", "text_anon", "llm"),
    "M3": ("corpus.cosine.t40.jsonl", "text_anon", "cosine-τ40"),
    "M4": ("corpus.cosine.t50.jsonl", "text_anon", "cosine-τ50"),
    "M5": ("corpus.cosine.t55.jsonl", "text_anon", "cosine-τ55"),
}

MODEL = "unsloth/Llama-3.2-1B"
USE_BF16 = False

EPOCHS, LORA_R, MAX_LEN, BATCH, ACCUM, LR, SEED = 8, 64, 512, 8, 2, 2e-4, 0
N_HELDOUT = 200
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

ONLY =(
  ["M3", "M4", "M5"] # Set which condition you'd like to proceed for the finetuning
)

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/wikibio"
except (ImportError, ModuleNotFoundError):
    SAVE_DIR = "/content/wikibio"
os.makedirs(SAVE_DIR, exist_ok=True)
print("output ->", SAVE_DIR)

Mounted at /content/drive
output -> /content/drive/MyDrive/wikibio


## Split

In [ ]:
_REPS = re.compile(r"^canary-\d+-r(\d+)$")

def reps_of(i):
    m = _REPS.match(i or "")
    return int(m.group(1)) if m else 1

is_canary = lambda r: r["id"].startswith("canary-")

def split(path, field):
    recs = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    recs = [r for r in recs if (r.get(field) or "").strip()]

    canaries = [r for r in recs if is_canary(r)]
    real = sorted((r for r in recs if not is_canary(r)), key=lambda r: r["id"])
    random.Random(SEED).shuffle(real)

    heldout, train_recs = real[:N_HELDOUT], real[N_HELDOUT:]
    train_recs += [r for r in canaries for _ in range(reps_of(r["id"]))]
    random.Random(SEED).shuffle(train_recs)

    by_id = {r["id"]: r for r in canaries}
    secrets = json.load(open(f"{SAVE_DIR}/canary_secrets.json"))

    measurable = {c for c in secrets if secrets[c]["secret"] in by_id[c][field]}
    return train_recs, heldout, measurable

## Training

Il modello base viene ricaricato da zero a ogni condizione (*M0/M1/M2...*)

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling


def _train(sigla, path, field, label):
    out_dir = f"{SAVE_DIR}/adapter-{sigla}"
    train_recs, heldout, measurable = split(f"{SAVE_DIR}/{path}", field)

    model, tok = FastLanguageModel.from_pretrained(
        model_name=MODEL, max_seq_length=MAX_LEN, dtype=None, load_in_4bit=True
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=2 * LORA_R,
        lora_dropout=0.0,
        bias="none",
        target_modules=TARGETS,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def encode(b):
        # Aggiungo EOS token
        return tok(
            [t + tok.eos_token for t in b[field]], truncation=True, max_length=MAX_LEN
        )

    ds = Dataset.from_list([{field: r[field]} for r in train_recs]).map(
        encode, batched=True, remove_columns=[field]
    )

    trainer = Trainer(
        model=model,
        train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
        args=TrainingArguments(
            output_dir=f"out-{sigla}",
            num_train_epochs=EPOCHS,
            learning_rate=LR,
            per_device_train_batch_size=BATCH,
            gradient_accumulation_steps=ACCUM,
            warmup_steps=20,
            logging_steps=50,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            fp16=not USE_BF16,
            bf16=USE_BF16,
            seed=SEED,
            save_strategy="no",
            report_to="none",
        ),
    )
    stats = trainer.train()

    tmp = f"adapter-{sigla}"
    model.save_pretrained(tmp)
    tok.save_pretrained(tmp)
    shutil.copytree(tmp, out_dir, dirs_exist_ok=True)

    json.dump(
        {
            "condition": sigla,
            "label": label,
            "source": os.path.basename(path),
            "model": MODEL,
            "field": field,
            "epochs": EPOCHS,
            "lora_r": LORA_R,
            "seed": SEED,
            "use_bf16": USE_BF16,
            "n_train_rows": len(train_recs),
            "n_heldout": len(heldout),
            "final_loss": round(stats.training_loss, 4),
            "measurable": sorted(measurable),
        },
        open(f"{SAVE_DIR}/train_{sigla}.json", "w"),
        indent=2,
    )

    print(f"[{sigla}] {label}: loss {stats.training_loss:.4f} -> {out_dir}")
    del model, trainer, ds
    gc.collect()
    torch.cuda.empty_cache()


for sigla in ONLY:
    path, field, label = REGISTRY[sigla]
    print(f"\n=== {sigla} — {label} ===")
    _train(sigla, path, field, label)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

=== M3 — cosine-τ40 ===
==((====))==  Unsloth 2026.9.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.9 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Map:   0%|          | 0/2210 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,210 | Num Epochs = 8 | Total steps = 1,112
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 45,088,768 of 1,280,903,168 (3.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


...

Map:   0%|          | 0/2210 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,210 | Num Epochs = 8 | Total steps = 1,112
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 45,088,768 of 1,280,903,168 (3.52% trained)


Step,Training Loss
50,2.009177
100,1.628004
150,1.581868
200,1.303809
250,1.303479
300,1.163349
350,0.950531
400,0.965979
450,0.736426
500,0.635029


Unsloth: Restored added_tokens_decoder metadata in adapter-M5/tokenizer_config.json.


[M5] cosine-τ55: loss 0.7009 -> /content/drive/MyDrive/wikibio/adapter-M5


In [ ]:
print("Modelli finetuned presenti:")
for sigla, (_, _, label) in REGISTRY.items():
    p = f"{SAVE_DIR}/adapter-{sigla}"
    if os.path.isdir(p):
        print(f"  {sigla:4s} {label}")

Modelli finetuned presenti:
  M0   raw
  M1   presidio
  M2   llm
  M3   cosine-τ40
  M4   cosine-τ50
  M5   cosine-τ55
